# Download PRISM Climate Data for Study Area

In this notebook, the study area boundary created in Notebook 01 will be used to download and save PRISM climate data for future use. 

**Add some info on the PRISM dataset and links**


## Step 1: Import Libraries and set Project Directories

In [5]:
# import libraries

# File management
import os
import pathlib
#import pyarrow # saving GeoParquet files

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # for SNOTEL API access
import zipfile

# Data Management
import pandas as pd
import xarray as xr

# Geospatial Management
import geopandas as gpd
from pygeohydro import WBD # site boundary based on watersheds
#import rasterio
#import rioxarray as rxr
#from shapely.geometry import box # lat/lon boundary boxes
#from shapely.ops import split
#from shapely.ops import unary_union # for intersecting boundaries

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas


In [2]:
import sys
from pathlib import Path

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
from src.data_download import prism

# once I'm using environment.yml I can get rid of the above steps

# from src
from src.data_download import prism

In [3]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Calculate Water Years

This step will calculate daily dates for the range of water years required.

In [4]:
# calculate water year dates

wy_dates = prism.water_year_dates()
wy_dates

[datetime.datetime(1989, 10, 1, 0, 0),
 datetime.datetime(1989, 10, 2, 0, 0),
 datetime.datetime(1989, 10, 3, 0, 0),
 datetime.datetime(1989, 10, 4, 0, 0),
 datetime.datetime(1989, 10, 5, 0, 0),
 datetime.datetime(1989, 10, 6, 0, 0),
 datetime.datetime(1989, 10, 7, 0, 0),
 datetime.datetime(1989, 10, 8, 0, 0),
 datetime.datetime(1989, 10, 9, 0, 0),
 datetime.datetime(1989, 10, 10, 0, 0),
 datetime.datetime(1989, 10, 11, 0, 0),
 datetime.datetime(1989, 10, 12, 0, 0),
 datetime.datetime(1989, 10, 13, 0, 0),
 datetime.datetime(1989, 10, 14, 0, 0),
 datetime.datetime(1989, 10, 15, 0, 0),
 datetime.datetime(1989, 10, 16, 0, 0),
 datetime.datetime(1989, 10, 17, 0, 0),
 datetime.datetime(1989, 10, 18, 0, 0),
 datetime.datetime(1989, 10, 19, 0, 0),
 datetime.datetime(1989, 10, 20, 0, 0),
 datetime.datetime(1989, 10, 21, 0, 0),
 datetime.datetime(1989, 10, 22, 0, 0),
 datetime.datetime(1989, 10, 23, 0, 0),
 datetime.datetime(1989, 10, 24, 0, 0),
 datetime.datetime(1989, 10, 25, 0, 0),
 datetime

## Step 3: Point Location Download for ML training

The PRISM data storage architecture doesn't allow for downloading data for a specific geographic area - you can either download point location information, or download the rasters for the entire country. So, I'll use the point location first to get access to the points I need for the SWE ML model, and then download the full rasters later.

Doesn't look like this is doable from the API. Either download manually, or download the whole raster set and go from there.

In [ ]:
variables = ['ppt', 'tmin', 'tmax']

url = "https://prism.oregonstate.edu/explorer/bulk.php"  # verify vs single-point
stats_str = " ".join(variables)  # "ppt tmax tmin"
    
start_date = 19951001
end_date = 19960601
wy = 1996

params = {
    "stats": stats_str,
    "units": "si",
    "range": "daily",
    "start": 19951001,   # YYYYMMDD
    "end": 19960601,
    "stability": "Unlikely to change",
    "lon": round(-111.95902, 5),
    "lat": round(45.59723, 5),
    "elev": round(8500.0 * 0.3048),
    #"call": "pp/daily_timeseries",
    "proc": "gridserv",
    "spares": "4km",
    "interp": "0",
}
            

response = requests.get(url, params=params, timeout=60)
# Try GET first since params look like query params, not form fields
# If GET fails, switch to requests.post()
response.raise_for_status()

# Single test request
print(response.text[:500])
print(response.headers.get("Content-Type"))


<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" lang="en">
  <head>
    <meta name="google-site-verification" content="fpm-ihn-NqE5SPhSZVC5R7CAIp41BOTLKm8FjrJT7A0" />
    <meta name="msvalidate.01" content="1E6248CC83F622D09CD63D4B759AB6F1" />
    <meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />

    <link type="image/x-icon" rel="shortcut icon" href="https://prism.oregonstate.edu/inc/images/favicon.ico" />
    <link type="text/css" rel="stylesheet" href="https:/
text/html; charset=UTF-8


## Step 3: Download PRISM rasters for ML training and application

In [ ]:
# set dirs here

# download rasters

# mask and clip w/ boundary?